# PerovStats main notebook
### This notebook is the main one to use when running PerovStats. It can take a directory containing multiple images and process them all in one run.

If you encounter bugs please make a report of them as a [GitHub issue](https://github.com/AFM-SPM/PerovStats/issues/new?template=bug_report.yml). Please provide as much info as you can, if you don't know some specific details that is ok.

Alternatively, email t.allwood@sheffield.ac.uk - I will get back to you as soon as I can

#### Before running this notebook please read the installation instructions (in the `/docs/` folder) to make sure the project dependencies are set up for the program.

#### To ensure the sliders work correctly please run this notebook one cell at a time rather than clicking the 'Run All' button

In [2]:
%load_ext autoreload
%autoreload 2

import os
import sys
import time

from loguru import logger
from pathlib import Path
import yaml
from topostats.io import LoadScans
import pandas as pd
from ipywidgets import interact, fixed, FloatSlider, Layout
import matplotlib.pyplot as plt
import matplotlib.cm as cm

sys.path.append(os.path.abspath(os.path.join('..')))
from src.perovstats.core.classes import PerovStats, ImageData
from src.perovstats.core.io import save_to_csv, save_config, save_images
from src.perovstats.core.image_processing import normalise_array
from src.perovstats.cli import setup_logger, deep_merge
from src.perovstats.fourier import split_frequencies
from src.perovstats.segmentation import segment_image
from src.perovstats.processing import format_time, completion_message
from src.perovstats.smears import find_smear_areas
from src.perovstats.grains import find_grains

---
#### These options can be changed to suit your needs. The `output_dir` folder will contain a selection of images from various stages in the process as well as the `.csv` files generated.

`img_files` should be a directory containing the AFM files (e.g. `.spm`), not the actual file itself.

If you have a custom config file you would like to use also enter the path into the `config_path` variable. If not, the default config is already loaded in ready for you.

In [3]:
base_dir = Path("path/to/folder/with/files(s)")
output_dir = Path("folder/to/save/data/to")
config_path = Path("../src/perovstats/default_config.yaml")

base_dir = Path("../example images")
output_dir = Path("./output")
config_path = Path("../src/perovstats/default_config.yaml")

#### Tweak parameters if necessary

##### 1) Identify the file type of the scan(s) used.

##### 2) Disable frequency splitting if necessary with `run_splitting = False`

##### 3) Adjust the cutoff bounds if no cutoff frequency can be found (if one is found its value is logged in the output below)

##### 4) Choose the min_rms value (affects isolation of perovskite grains from silicon topology)

##### 5) Select the segmentation method to use
##### As a reminder:
- `traditional`:
    - hard-coded
    - requires a configuration option (`threshold_offset`)
    - faster (a few seconds/image)
    - will contain inaccuracies
- `cellpose`:
    - machine learning
    - no manual configuration required
    - slower (a few minutes/image) - unless your device uses an Nvidia GPU
    - more accurate

In [4]:
# The file extension of the input image(s).
file_ext = ".spm"

# If you are inputting images without a background material frequency splitting can be disabled here.
run_splitting = True # Options: True, False

# Adjust these bounds to increase the difference if no cutoff can be found
cutoff_bounds = [0, 0.3]

# Adjust this value if the background material is still visible after frequency splitting or the image is too flat.
# DECREATE if the image still contains the background material
# INCREASE if the image is too flat
min_rms = 10


segmentation_method = "traditional" # Options: traditional, cellpose

# If 'traditional' segmentation has been chosen: (this can be adjusted with a slider further down the notebook)
threshold_offset = -0.5

# Comment or uncomment image names to select which ones to output
image_set = [
    "highpass_mask",
    "highpass",
    "lowpass",
    "mask",
    "numbered_grains",
    "original_mask",
    "original",
    "rgb_grains",
    "smears",
]


---
#### Here the config and images will be loaded in ready for the main process

In [5]:
setup_logger()

default_config_path = Path("../src/perovstats/default_config.yaml")
with default_config_path.open() as f:
    config = yaml.safe_load(f)

if config_path.resolve() != default_config_path.resolve():
    if config_path.exists():
        with config_path.open() as f:
            custom_config = yaml.safe_load(f)

        config = deep_merge(config, custom_config)
    else:
        logger.error("Error: custom configuration file could not be found. Please check the set filepath above.")

config["output"]["image_set"] = image_set
config["file_ext"] = file_ext

config["fourier"]["run"] = run_splitting
config["fourier"]["cutoff_bounds"] = cutoff_bounds
config["fourier"]["min_rms"] = min_rms

config["segmentation"]["segmentation_method"] = segmentation_method
config["segmentation"]["traditional"]["threshold_offset"] = threshold_offset

file_ext = config["file_ext"]
img_files = list(Path(base_dir).glob("*" + file_ext))

time_start = time.perf_counter()

# Load scans
load_config = config["loading"]
loadscans = LoadScans(img_files, config)
try:
    loadscans.get_data()
except ValueError as e:
    logger.warning(e)
    logger.warning(f"Channel {load_config['channel']} not found in file. Please ensure the config option is correct and all files contain the required channel.")
image_dicts = loadscans.img_dict

# Create the dataclasses for the whole process and for each image found
perovstats_object = PerovStats(config=config, images=[])
for filename, topostats_object in image_dicts.items():
    image_data = ImageData(
        success=True,
        filename=filename,
        pixel_to_nm_scaling=topostats_object.pixel_to_nm_scaling,
        image_original=topostats_object.image_original,
        image_flattened=None)
    perovstats_object.images.append(image_data)

logger.info("----------------------------------------------------------")
logger.info(f"Loaded {len(perovstats_object.images)} images")


[Wed, 03 Jun 2026 14:56:59] [INFO    ] [topostats] Extracting image from ..\example images\4_c60_perovonsil_ref_10um.PFQNM.spm
14:56:59 | INFO    | spm.py          | Loading image from : ..\example images\4_c60_perovonsil_ref_10um.PFQNM.spm
14:56:59 | INFO    | spm.py          | [4_c60_perovonsil_ref_10um.PFQNM] : Loaded image from : ..\example images\4_c60_perovonsil_ref_10um.PFQNM.spm
14:57:00 | INFO    | spm.py          | [4_c60_perovonsil_ref_10um.PFQNM] : Extracted channel Height
14:57:00 | INFO    | spm.py          | [4_c60_perovonsil_ref_10um.PFQNM] : Pixel to nm scaling : 19.53125
[Wed, 03 Jun 2026 14:57:00] [INFO    ] [topostats] Extracting image from ..\example images\TR_5um_ST4_14_26_per_Si_tapp4.0_00010.spm
14:57:00 | INFO    | spm.py          | Loading image from : ..\example images\TR_5um_ST4_14_26_per_Si_tapp4.0_00010.spm
14:57:00 | INFO    | spm.py          | [TR_5um_ST4_14_26_per_Si_tapp4.0_00010] : Loaded image from : ..\example images\TR_5um_ST4_14_26_per_Si_tapp4.0_

---
#### This is the main PerovStats process. Here images will be cycled through and processed one by one. You can see updates on its progress from the logs appearing under the code block

When mask segmentation completes for an image warnings may appear. As long as the green `SUCCESS` log appears under it you can ignore these warnings.

Once this cell has been run a slider will appear for each image. These can be adjusted and a preview of the split images will appear. Once you are satisfied with the slider values you can move on to the next cell.

In [ ]:
cmap = config["colour_scheme"]

def preview_split(image_object, split_freq):
    split_frequencies(perovstats_object.config, image_object, split_freq=split_freq)

    # Show the split images
    _, ax = plt.subplots(1, 2)

    ax[0].imshow(image_object.high_pass, cmap=cmap)
    ax[0].set_title("High passed image (Perovskite)")
    ax[0].axis("off")

    ax[1].imshow(image_object.low_pass, cmap=cmap)
    ax[1].set_title("Low passed image (Silicon)")
    ax[1].axis("off")

    plt.tight_layout()
    plt.show()


for idx, image_object in enumerate(perovstats_object.images):
    print(f"--- Image {idx + 1} ---")
    # Apply fourier transform to split the image into a low-passed and high-passed image
    split_frequencies(perovstats_object.config, image_object)

    interact(
        preview_split,
        image_object=fixed(image_object),
        split_freq=FloatSlider(
            min=0.001,
            max=0.5,
            step=0.001,
            value=image_object.cutoff,
            description='Cutoff Frequency:',
            layout=Layout(width='60%'),
            readout_format='.3f',
            continuous_update=False
        )
    )

--- Image 1 ---
14:57:07 | INFO    | fourier.py      | [4_c60_perovonsil_ref_10um.PFQNM.spm] : *** Frequency splitting ***
14:57:08 | INFO    | fourier.py      | [4_c60_perovonsil_ref_10um.PFQNM.spm] : Frequency cutoff: 0.1295 (301.6591nm)
14:57:08 | INFO    | fourier.py      | [4_c60_perovonsil_ref_10um.PFQNM.spm] : Splitting image frequencies


interactive(children=(FloatSlider(value=0.1294921875, continuous_update=False, description='Cutoff Frequency:'…

--- Image 2 ---
14:57:08 | INFO    | fourier.py      | [TR_5um_ST4_14_26_per_Si_tapp4.0_00010.spm] : *** Frequency splitting ***
14:57:09 | INFO    | fourier.py      | [TR_5um_ST4_14_26_per_Si_tapp4.0_00010.spm] : Frequency cutoff: 0.0861 (226.7574nm)
14:57:09 | INFO    | fourier.py      | [TR_5um_ST4_14_26_per_Si_tapp4.0_00010.spm] : Splitting image frequencies


interactive(children=(FloatSlider(value=0.08613281249999999, continuous_update=False, description='Cutoff Freq…

If traditional segmentation is being used then similar to above a slider will appear letting you adjust the sensitivity of the algorithm.

In [ ]:
get_cmap = cm.get_cmap(cmap)

def preview_segmentation(image_object, threshold_offset):
    perovstats_object.config["segmentation"]["traditional"]["threshold_offset"] = threshold_offset
    segment_image(perovstats_object.config, image_object)

    mask_overlay_trad = image_object.high_pass
    norm_highpass = normalise_array(mask_overlay_trad)
    rgba_highpass = get_cmap(norm_highpass)
    mask_overlay_trad = rgba_highpass[..., :3]
    mask_overlay_trad[image_object.mask > 0] = [0, 0, 1]

    # Show the split images
    _, ax = plt.subplots(1, 2)

    ax[0].imshow(mask_overlay_trad, cmap=cmap)
    ax[0].set_title("Overlayed mask")
    ax[0].axis("off")

    ax[1].imshow(image_object.mask, cmap=cmap)
    ax[1].set_title("Mask")
    ax[1].axis("off")

    plt.tight_layout()
    plt.show()

for idx, image_object in enumerate(perovstats_object.images):
    logger.info("----------------------------------------------------------")
    logger.info(f"Processing {image_object.filename} ({idx+1}/{len(perovstats_object.images)})")
    logger.info("----------------------------------------------------------")
    logger.debug(f"[{image_object.filename}] : Image dimensions: {image_object.image_original.shape}")
    logger.debug(f"[{image_object.filename}] : pixel_to_nm_scaling: {image_object.pixel_to_nm_scaling}")

    # If frequency splitting was run and failed skip processing on the rest of the image
    if not image_object.success:
        continue
    
    # Generate segmented mask of the high-passed image
    segment_image(perovstats_object.config, image_object)

    if config["segmentation"]["segmentation_method"] == "traditional":
        interact(
            preview_segmentation,
            image_object=fixed(image_object),
            threshold_offset=FloatSlider(
                min=-3,
                max=3,
                step=0.01,
                value=perovstats_object.config["segmentation"]["traditional"]["threshold_offset"],
                description='Threshold offset:',
                layout=Layout(width='60%'),
                readout_format='.3f',
                continuous_update=False,
            )
        )

14:58:00 | INFO    | 290838068.py    | ----------------------------------------------------------
14:58:00 | INFO    | 290838068.py    | Processing 4_c60_perovonsil_ref_10um.PFQNM.spm (1/2)
14:58:00 | INFO    | 290838068.py    | ----------------------------------------------------------
14:58:00 | DEBUG   | 290838068.py    | [4_c60_perovonsil_ref_10um.PFQNM.spm] : Image dimensions: (512, 512)
14:58:00 | DEBUG   | 290838068.py    | [4_c60_perovonsil_ref_10um.PFQNM.spm] : pixel_to_nm_scaling: 19.53125
14:58:00 | INFO    | segmentation.py | [4_c60_perovonsil_ref_10um.PFQNM.spm] : *** Mask creation ***
14:58:00 | INFO    | segmentation.py | [4_c60_perovonsil_ref_10um.PFQNM.spm] : Creating grain mask


C:\Users\tobya\AppData\Local\Temp\ipykernel_34784\290838068.py:1: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  get_cmap = cm.get_cmap(cmap)


14:58:05 | SUCCESS | segmentation.py | [4_c60_perovonsil_ref_10um.PFQNM.spm] : Mask created successfully.
14:58:05 | INFO    | pruning.py      | [4_c60_perovonsil_ref_10um.PFQNM.spm] : *** Pruning ***


interactive(children=(FloatSlider(value=-0.5, continuous_update=False, description='Threshold offset:', layout…

14:58:18 | INFO    | 290838068.py    | ----------------------------------------------------------
14:58:18 | INFO    | 290838068.py    | Processing TR_5um_ST4_14_26_per_Si_tapp4.0_00010.spm (2/2)
14:58:18 | INFO    | 290838068.py    | ----------------------------------------------------------
14:58:18 | DEBUG   | 290838068.py    | [TR_5um_ST4_14_26_per_Si_tapp4.0_00010.spm] : Image dimensions: (512, 512)
14:58:18 | DEBUG   | 290838068.py    | [TR_5um_ST4_14_26_per_Si_tapp4.0_00010.spm] : pixel_to_nm_scaling: 9.765625
14:58:18 | INFO    | segmentation.py | [TR_5um_ST4_14_26_per_Si_tapp4.0_00010.spm] : *** Mask creation ***
14:58:18 | INFO    | segmentation.py | [TR_5um_ST4_14_26_per_Si_tapp4.0_00010.spm] : Creating grain mask
14:58:26 | SUCCESS | segmentation.py | [TR_5um_ST4_14_26_per_Si_tapp4.0_00010.spm] : Mask created successfully.
14:58:26 | INFO    | pruning.py      | [TR_5um_ST4_14_26_per_Si_tapp4.0_00010.spm] : *** Pruning ***


In [ ]:
for idx, image_object in enumerate(perovstats_object.images):
    # Find smear areas to be ignored/ removed
    find_smear_areas(perovstats_object.config, image_object)

    # Identify individual grains from mask and generate statistics on them
    find_grains(perovstats_object.config, image_object)

    logger.info(f"[{image_object.filename}] : *** Exporting data ***")
    # Save image and grain data to their own .csv file
    image_df = pd.DataFrame([image_object.to_dict()])
    grains_list = []
    for grain in image_object.grains.values():
        grains_list.append(grain.to_dict())
    grain_df = pd.DataFrame(grains_list)

    save_images(perovstats_object.config, image_object)

    output_filename = f"{output_dir}/{image_object.filename}/image_statistics.csv"
    save_to_csv(image_df, output_filename)
    output_filename = f"{output_dir}/{image_object.filename}/grain_statistics.csv"
    save_to_csv(grain_df, output_filename)
    # Save the config settings in a .yaml
    output_filename = Path(output_dir) / Path(image_object.filename) / "config.yaml"
    save_config(perovstats_object.config, output_filename)

    logger.info(
        f"[{image_object.filename}] : Exported data and config to {Path(output_dir) / Path(image_object.filename)}",
    )

---
#### Once the process has been completed, basic stats about the run will be shown below.

Data about individual images will be saved in the `output_dir` you selected at the top with one subfolder for each image.

In [ ]:
time_end = time.perf_counter()
time_taken = format_time(time_end - time_start)
time_per_image = format_time((time_end - time_start) / len(perovstats_object.images))
completion_message(perovstats_object, time_taken, time_per_image)